# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level dataset metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}\n")

## 2. Data Overview
Let's review available record sets, their fields, and associated `@id`s. We'll use these `@id`s for further data access.

In [ ]:
# List all available record sets by their @id
record_sets = list(dataset.record_sets())

print("Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs.get('@id'))}")

# For each record set, show its fields and their @id
for rs in record_sets:
    print(f"\nFields for Record Set {rs['@id']}: {rs.get('name', rs.get('@id'))}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        if isinstance(fld, dict):
            print(f"  - {fld.get('@id', 'unknown_id')}: {fld.get('name', fld.get('@id', 'unknown'))}")
        else:
            print(f"  - {fld}")

## 3. Data Extraction
Let's load the data for each record set into a DataFrame. Use record set and field `@id`s as discovered above.

If the dataset contains multiple record sets, they will be available in the `record_sets` variable.

In [ ]:
# Extract data for each record set
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    # List records
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}. Columns: {df.columns.tolist()}")
    else:
        print(f"No records loaded for record set {rs_id}.")

# As an example, show one DataFrame, if any available
if dataframes:
    example_rsid = next(iter(dataframes.keys()))
    print(f"\nShowing first five records for record set {example_rsid}:")
    display(dataframes[example_rsid].head())

## 4. Exploratory Data Analysis (EDA)
In this section, we conduct simple EDA, such as filtering, normalization, and grouping operations, all identified via `@id` fields.

Please adapt the `numeric_field_id` and `group_field_id` below to reference actual fields from your chosen record set.

In [ ]:
# Pick a record set for EDA (use the one previously examined)
if dataframes:
    record_set_id = example_rsid
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")

    # Choose a numeric field by its @id (Change as per actual fields)
    # For example, suppose the field @id is 'log_likelihood', adjust as necessary
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if 'likelihood' in col.lower() or 'age' in col.lower() or 'iteration' in col.lower():
            numeric_field_id = col
        if 'ward' in col.lower() or 'group' in col.lower() or 'county' in col.lower():
            group_field_id = col
    if not numeric_field_id:
        print("No obvious numeric field found. Please set 'numeric_field_id' to a column from: ", df.columns.tolist())
    else:
        # Remove outliers: e.g., values > threshold (here, arbitrarily 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized" ]].head())

        # Group by a group/categorical field, if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found in columns:", df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields. Update field identifiers as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot by group, if group field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded, reviewed, and performed basic exploration of the FAIR² dataset using the `mlcroissant` library, referencing all entities via their Croissant `@id` fields.

- Use `mlcroissant` for dataset structure discovery.
- Use `@id` references for robust, schema-compliant data extraction and analysis.
- Further steps may include advanced statistical tests or model fitting, depending on the research goal.